# TALON Tutorial: Analysis, Profiling & Deployment

```bash
pip install t1c-talon
```

In [ ]:
import os, numpy as np, torch, torch.nn as nn
from talon import ir, bridge, viz, sdk
os.makedirs("models", exist_ok=True)
print(f"SDK: {sdk.__version__}")

## 1. Model Conversion

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16*14*14, 10)
    def forward(self, x):
        return self.fc(self.flatten(self.pool(self.relu(self.conv1(x)))))

model = SimpleCNN()
sample = torch.randn(1, 1, 28, 28)
graph = bridge.to_ir(model, sample)
print(f"Exported: {len(graph.nodes)} nodes")
ir.write("models/simple_cnn.t1c", graph)

In [ ]:
executor = bridge.ir_to_torch(graph)
with torch.no_grad():
    diff = (model(sample) - executor(sample)).abs().max().item()
print(f"Round-trip diff: {diff:.2e}, Equivalent: {diff < 1e-5}")

## 2. Analysis

In [ ]:
stats = sdk.analyze_graph(graph)
print(f"Nodes: {stats.node_count}, Params: {stats.total_params:,}")
for nt, c in stats.type_counts.items():
    print(f"  {nt}: {c}")

In [ ]:
for i, p in enumerate(sdk.trace_path(graph, "input", "output")):
    print(f"Path {i+1}: {" -> ".join(p)}")

## 3. Profiling

In [ ]:
prof = sdk.profile_graph(graph)
print(f"Memory: {prof.total_memory:,} bytes, MACs: {prof.mac_ops:,}")
for r in prof.recommendations:
    print(f"  - {r}")

## 4. Linting

In [ ]:
res = sdk.lint_graph(graph)
print(f"Valid: {res.is_valid}, Warnings: {len(res.warnings)}")
for w in res.warnings:
    print(f"  [{w.code}] {w.message}")

## 5. Fingerprinting

In [ ]:
fp = sdk.fingerprint_graph(graph)
print(f"Fingerprint: {fp[:32]}...")
print(f"Deterministic: {fp == sdk.fingerprint_graph(graph)}")